## Submission instructions

All code that you write should be in this notebook. Please include your names and student numbers. You have to submit this notebook, with your code and answers filled in. Make sure to add enough documentation.

For questions, make use of the "Lab" session (see schedule).
Questions can also be posted to the MS teams channel called "Lab".

**Note:** You are free to make use of Python libraries (e.g., numpy, sklearn, etc.) except any *fairness* libraries.

#### Name and student numbers
*Anastasios Karampetsios* (a.karampetsios@students.uu.nl)
- Student Number: 4931858

*David van Nistelrooij* (d.y.vieiradecarvalhovannistelrooij@students.uu.nl)
- Student Number: 9517835


## Dataset

In this assignment we are going to use the **COMPAS** dataset.

If you haven't done so already, take a look at this article: https://www.propublica.org/article/machine-bias-risk-assessments-in-criminal-sentencing.
For background on the dataset, see https://www.propublica.org/article/how-we-analyzed-the-compas-recidivism-algorithm.

**Reading in the COMPAS dataset**

The dataset can be downloaded here: https://github.com/propublica/compas-analysis/blob/master/compas-scores-two-years.csv

For this assignment, we focus on the protected attribute *race*.

The label (the variable we want to be able to predict) represents recidivism, which is defined as a new arrest within 2 years.

In [ ]:
!wget -c https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv

In [4]:
import pandas as pd
compas_data = pd.read_csv('compas-scores-two-years.csv')

We apply several data preprocessing steps, including only retaining Caucasians and African Americans.

In [5]:
# Filter rows
compas_data = compas_data[(compas_data.days_b_screening_arrest <= 30)
            & (compas_data.days_b_screening_arrest >= -30)
            & (compas_data.is_recid != -1)
            & (compas_data.c_charge_degree != 'O')
            & (compas_data.score_text != 'N/A')
            # Only retaining Caucasians and African-Americans
            & ((compas_data.race == 'Caucasian') | (compas_data.race == 'African-American'))]

# Remove columns related to COMPAS scores
compas_data = compas_data.loc[:, ~(
            compas_data.columns.str.endswith('.1') | # Remove duplicate columns
            compas_data.columns.str.startswith('r_') |
            compas_data.columns.str.startswith('v_') |
            compas_data.columns.str.startswith('vr_') |
            compas_data.columns.isin(['is_violent_recid','violent_recid', 'is_recid', 'name', 'first',
                                   'last', 'compas_screening_date', 'decile_score', 'days_b_screening_arrest',
                                   'c_days_from_compas', 'type_of_assessment', 'score_text', 'screening_date',
                                   'in_custody', 'out_custody', 'start', 'end', 'event', 'c_arrest_date'])
)]

Take a look at the data:

In [ ]:
print(compas_data.head())

    id     sex         dob  age       age_cat              race  \
1    3    Male  1982-01-22   34       25 - 45  African-American   
2    4    Male  1991-05-14   24  Less than 25  African-American   
6    8    Male  1974-07-23   41       25 - 45         Caucasian   
8   10  Female  1976-06-03   39       25 - 45         Caucasian   
10  14    Male  1988-06-01   27       25 - 45         Caucasian   

    juv_fel_count  juv_misd_count  juv_other_count  priors_count  \
1               0               0                0             0   
2               0               0                1             4   
6               0               0                0            14   
8               0               0                0             0   
10              0               0                0             0   

              c_jail_in           c_jail_out  c_case_number c_offense_date  \
1   2013-01-26 03:45:27  2013-02-05 05:36:53  13001275CF10A     2013-01-26   
2   2013-04-13 04:58:34  2013-04

Below are short descriptions for each column in the COMPAS dataset that we have decided to keep for this assignment.

- `id`: unique identifier for a person/record
- `sex`: recorded sex of the person
- `dob`: date of birth
- `age`: age at the time of screening
- `age_cat`: age category/range
- `race`: race category
- `juv_fel_count`: number of juvenile felony charges
- `juv_misd_count`: number of juvenile misdemeanor charges
- `juv_other_count`: number of other juvenile charges
- `priors_count`: number of prior convictions/offenses
- `c_jail_in` and `c_jail_out`: jail entry and release for the current case
- `c_case_number`: current case number
- `c_offense_date`: offense date for the current case
- `c_charge_degree`: charge degree for the current case (`F` = felony; `M` = misdemeanor)
- `c_charge_desc`: text description of the current charge


Now take a look at the distribution of the protected attribute `race` and the distribution of our outcome variable `two_year_recid`.

**Note:** in the context of fair machine learning, the favorable label here is no recidivism, i.e., ```two_year_recid = 0```. So think about how what you will code as the positive class in your machine learning experiments, and make sure your interpretation of the results is consistent with this.

In [6]:
print('Number of instances per race category:')
print(compas_data[['race', 'two_year_recid']].value_counts())

Number of instances per race category:
race              two_year_recid
African-American  1                 1661
                  0                 1514
Caucasian         0                 1281
                  1                  822
Name: count, dtype: int64


## Data analysis

### **1. Exploration**

First we perform an exploratory analysis of the data.

**Question:** What is the size of the data? (i.e. how many data instances does it contain?)


In [7]:
print(len(compas_data))

#above cell = 1661 + 1514+ 1281+ 822 = 5278

5278


**Question:** In the dataset, the protected attribute is `race`, which has two categories: White and African Americans. How many data instances belong to each category?

In [8]:
# Your code
total = compas_data['race'].value_counts()
print(total)

race
African-American    3175
Caucasian           2103
Name: count, dtype: int64


**Question:** What are the base rates (the probability of a favorable outcome for the two protected attribute classes)?

In [10]:
# Your code
# Group by "race"
# Count how many in each group
# Count how many favorable in each group ("two_year_recid" == 0)
no_recidivism = compas_data['two_year_recid'] == 0
people_with_no_recidivism = compas_data[no_recidivism]
no_recidivism_race = people_with_no_recidivism['race'].value_counts()
print(no_recidivism_race)
base_rate = no_recidivism_race / total
print(base_rate)

race
African-American    1514
Caucasian           1281
Name: count, dtype: int64
race
African-American    0.47685
Caucasian           0.60913
Name: count, dtype: float64


**Question:** What are the base rates for the combination of both race and sex categories?

In [11]:
# Your code
no_recidivism_sex = people_with_no_recidivism[['race', 'sex']].value_counts()
total_race_sex = compas_data[['race', 'sex']].value_counts()

base_rate_race_sex = no_recidivism_sex / total_race_sex
print(base_rate_race_sex)



race              sex   
African-American  Male      0.444783
Caucasian         Male      0.597779
African-American  Female    0.630237
Caucasian         Female    0.647303
Name: count, dtype: float64


**Question**

Write down a short interpretation of the statistics you calculated. What do you see?
> Answer: About 64,7% of Caucasian Females did not recidivate. About 63% of African-American Females did not recidivate. About 59,8% of Caucasian Males did not recidivate and about 44,48% of African-American Males did not recidivate. Females generally have a higher favorable outcomes than males. African-American Males have the lowest favorable outcome rate in this dataset


### **2. Performance measures**

You will have to measure the performance and fairness of different classifiers in question 5. The performance will be calculated with the precision, recall, F1 and accuracy.
Additionally, you will have to calculate the statistical/demographic parity, the true positive rate (recall) and false positive rate per race group.

Make sure that you are able to calculate these metrics in the cell below.

In [52]:
# # Your code for the performance measures
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def measure (ground_truth, prediction, race_group):
  accuracy = accuracy_score(ground_truth, prediction)
  print(f"accuracy: {accuracy}")
  precision = precision_score(ground_truth, prediction, pos_label = 0)
  print(f"precision:{precision}")
  recall = recall_score(ground_truth, prediction, pos_label=0)
  print(f"recall: {recall}")
  f1 = f1_score(ground_truth, prediction, pos_label=0)
  print(f"f1: {f1}")
  print()

  groups = race_group.unique()
  for x in groups:
    print(f" group: {'Caucasian' if x else 'African-American'}")
    group_select = race_group == x
    ground_truth_race_group = ground_truth[group_select]
    prediction_race_group = prediction[group_select]
    predicted_no_recidivism = prediction_race_group == 0

    statistical_parity = predicted_no_recidivism.sum() / len(prediction_race_group)
    print(f" statistical parity: {statistical_parity}")

    correct_no_recidivism = (ground_truth_race_group == 0) & (prediction_race_group == 0)
    correct_no_recidivism_amount = correct_no_recidivism.sum()
    actual_no_recidivism = ground_truth_race_group == 0
    actual_no_recidivism_amount = actual_no_recidivism.sum()
    true_positive_rate = correct_no_recidivism_amount / actual_no_recidivism_amount
    print(f"true positive rate: {true_positive_rate}")

    incorrect_no_recidivism = (ground_truth_race_group == 1) & (prediction_race_group == 0)
    incorrect_no_recidivism_amount = incorrect_no_recidivism.sum()
    actual_recidivism = ground_truth_race_group == 1
    actual_recidivism_amount = actual_recidivism.sum()
    false_positive_rate = incorrect_no_recidivism_amount / actual_recidivism_amount
    print(f"false positive rate: {false_positive_rate}")
    print()


### **3. Prepare the data**
For the classifiers in question 5, the input of the model can only contain numerical values, it is therefore important to convert the strings in the columns (features) of interest of the `compas_data` to floats or integers.

The columns of interest are features that you think will be informative or interesting in predicting the outcome variable.
Use the cell below to explore which of the Compas variables you need to convert to be able to use them for the classifiers.

Generate a new dataframe with your selected features in the right encoding (also make sure to include `two_year_recid`). You can implement this yourself, or use the `LabelEncoder` from `sklearn`.

In [68]:
# Your code to prepare the data
import os

chosen_attributes = [
    "sex",
    "age",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "two_year_recid",
]

dataset = compas_data[chosen_attributes].copy().reset_index(drop=True)

# Encoding 'sex'
#   'Male'   ->  1
#   'Female' ->  0
dataset['sex'] = (dataset['sex'] == 'Male').astype(int)

# Encoding 'race'
#   'Caucasian'   ->  1
#   'African-American' ->  0
dataset['race'] = (dataset['race'] == 'Caucasian').astype(int)

# Encoding 'c_charge_degree'
#   'M'   ->  1
#   'F' ->  0
dataset['c_charge_degree'] = (dataset['c_charge_degree'] == 'M').astype(int)


os.makedirs("datasets", exist_ok=True)

dataset.to_csv("datasets/dataset.csv", index=False)





**Question**

Give a short motivation (one-two sentence) per feature why you think this is informative or interesting to take into account.
> Answer:
> - **'race'**, **'sex'** ->  protected attributes, required by the assignment. Also useful to evaluate intersectional fairness across demographic groups
> - **'age'**   -> Age can be informative about recidivism likelihood, since different age groups exhibit different recidivism patterns
> - **'juv_fel_count'**, **'juv_misd_count'**, **'juv_other_count'**, **'priors_count'**   -> Juvenile offence counts as well as total number of past offence capture a defendants prior criminal history and can be strongly correlated with the likelihood of recidivism
> - **'c_charge_degree'** -> Severity of charge degree can help determine likelihood of future re-offending

### **4. Train and test split**

Divide the dataset into a train (80%) and test split (20%), either by implementing it yourself, or by using an existing library.

**Note:** Usually when carrying out machine learning experiments,
we also need a dev set for developing and selecting our models (incl. tuning of hyper-parameters).
However, in this assignment, the goal is not to optimize
the performance of models so we'll only use a train and test split.




In [ ]:
# import sys
# !{sys.executable} -m pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl (36.5 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [69]:
# Your code to split the data
from sklearn.model_selection import train_test_split

dataset = pd.read_csv(f"datasets/dataset.csv")

x_features = [att for att in list(dataset.columns) if att != "two_year_recid"]
X = dataset[x_features].copy()
Y = dataset['two_year_recid'].copy()

# Split 1: Stratifying on Y only
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=100,
    stratify=Y
)
print(" - Stratify by: Y\n")
print(f"Recidivism Rate in train set: {Y_train.mean()}")
print(f"Recidivism Rate in test set: {Y_test.mean()}")


 - Stratify by: Y

Recidivism Rate in train set: 0.4703931785883467
Recidivism Rate in test set: 0.4706439393939394


### **5. Classifiers**

Now, train and test different classifiers and report the following statistics:

* Overall performance:

  * Precision
  * Recall
  * F1
  * Accuracy

* Fairness performance:

  * The statistical parity difference for the protected attribute `race`(i.e. the difference in the probability of receiving a favorable label between the two protected attribute groups);
  * The true positive rates of the two protected attribute groups
  * The false positive rates of the two protected attribute groups.

For training the classifier we recommend using scikit-learn (https://scikit-learn.org/stable/).

#### **5.1 Regular classification**
Train a logistic regression classifier with the race feature and all other features that you are interested in.

In [72]:
# Your code for classifier 1
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def create_and_train_LogReg(X_train, Y_train, X_test,  Y_test):
    model = LogisticRegression()
    model.fit(X_train, Y_train)
    test_pred = model.predict(X_test)

    return model

classifier_1 = LogisticRegression()
classifier_1.fit(X_train, Y_train)
test_pred_1 = classifier_1.predict(X_test)

measure(ground_truth=Y_test, prediction=test_pred_1, race_group=X_test['race'])



accuracy: 0.6628787878787878
precision:0.6629213483146067
recall: 0.738819320214669
f1: 0.6988155668358714

 group: African-American
 statistical parity: 0.4813614262560778
true positive rate: 0.6423841059602649
false positive rate: 0.326984126984127

 group: Caucasian
 statistical parity: 0.7425968109339408
true positive rate: 0.8521400778210116
false positive rate: 0.5879120879120879



#### **5.2 Without the protected attribute**
Train a logistic regression classifier without the race feature, but with all other features you used in 5.1.


In [71]:
# Your code for classifier 2
X_train_no_race = X_train.drop(columns='race').copy()

X_test_no_race = X_test.drop(columns='race').copy()

classifier_2 = LogisticRegression()
classifier_2.fit(X_train_no_race, Y_train)
test_pred_2 = classifier_2.predict(X_test_no_race)
measure(ground_truth=Y_test, prediction=test_pred_2, race_group=X_test['race'])

accuracy: 0.6619318181818182
precision:0.6650326797385621
recall: 0.7280858676207513
f1: 0.695132365499573

 group: African-American
 statistical parity: 0.49270664505672607
true positive rate: 0.6556291390728477
false positive rate: 0.33650793650793653

 group: Caucasian
 statistical parity: 0.7015945330296127
true positive rate: 0.8132295719844358
false positive rate: 0.5439560439560439



**Question**

Write down a short interpretation of the results you calculated. What do you see?
> Answer:

> **Overall Accuracy, Precision, Recall, F1:** relatively same

> * Removing the same attribute barely affects overall performance

> **Statistical Parity**

> * **before**: 0.743 - 0.481 = 0.262

> * **after**: 0.702 - 0.493 = 0.209

> * **'Caucasian'**: slightly lower base rate, TPR, FPR

> * **'African-American'**: slightly higher base rate, TPR, FPR


> * **Conclusion:** Removing the race attribute slightly reduces statistical parity gap in favor of the Caucasian Group. However there's still a gap between the Caucasian and the African-American groups, indicating that other existing features also reflect structural inequalities in the world leading to biased outcomes in favor of one group. Therefore removing the race feature does lead to slightly fairer predictions without sacrificing overall performance, but does not completely eradicate unfairness.




#### **5.3 Pre-processing: Reweighing**
Train and test a classifier with weights (see lecture slide for the weight calculation)

In [49]:
# Your code for classifier 3

# Concat Training set
training_set = pd.concat([X_train, Y_train], axis=1).reset_index(drop=True)

# Marginal Probabilities
p_caucasian = (training_set['race'] == 1).astype(int).mean()
p_afr_am = (training_set['race'] == 0).astype(int).mean()
p_recid = training_set['two_year_recid'].mean()
p_not_recid = 1 - p_recid

# Expected Probabilities
p_exp_cauc_recid = p_caucasian*p_recid
p_exp_cauc_not_recid = p_caucasian*p_not_recid
p_exp_afr_recid = p_afr_am*p_recid
p_exp_afr_not_recid = p_afr_am*p_not_recid

# Observed Probabilities
p_obs_cauc_recid = ((training_set['race'] == 1) & (training_set['two_year_recid'] == 1)).astype(int).mean()
p_obs_cauc_not_recid = ((training_set['race'] == 1) & (training_set['two_year_recid'] == 0)).astype(int).mean()
p_obs_afr_recid = ((training_set['race'] == 0) & (training_set['two_year_recid'] == 1)).astype(int).mean()
p_obs_afr_not_recid = ((training_set['race'] == 0) & (training_set['two_year_recid'] == 0)).astype(int).mean()

# Weights
w_cauc_recid = p_exp_cauc_recid / p_obs_cauc_recid
w_cauc_not_recid = p_exp_cauc_not_recid / p_obs_cauc_not_recid
w_afr_recid = p_exp_afr_recid / p_obs_afr_recid
w_afr_not_recid = p_exp_afr_not_recid / p_obs_afr_not_recid


print("MARGINAL PROBABILITIES\n")
print(f"p(race='Caucasian') = {p_caucasian:.4f}")
print(f"p(race='African-American') = {p_afr_am:.4f}")
print(f"p(two_year_recid=1) = {p_recid:.4f}")
print(f"p(two_year_recid=0) = {p_not_recid:.4f}")
print("\n")

print("Expected vs Observed Probabilities and Weights\n")
print("Race = 'Caucasian', Recidivate = True")
print(f"\tExpected: {p_exp_cauc_recid:.4f}")
print(f"\tObserved: {p_obs_cauc_recid:.4f}")
print(f"\tWeight: {w_cauc_recid:.4f}")
print()

print("Race = 'Caucasian', Recidivate = False")
print(f"\tExpected: {p_exp_cauc_not_recid:.4f}")
print(f"\tObserved: {p_obs_cauc_not_recid:.4f}")
print(f"\tWeight: {w_cauc_not_recid:.4f}")
print()

print("Race = 'African-American', Recidivate = True")
print(f"\tExpected: {p_exp_afr_recid:.4f}")
print(f"\tObserved: {p_obs_afr_recid:.4f}")
print(f"\tWeight: {w_afr_recid:.4f}")
print()


print("Race = 'African-American', Recidivate = False")
print(f"\tExpected: {p_exp_afr_not_recid:.4f}")
print(f"\tObserved: {p_obs_afr_not_recid:.4f}")
print(f"\tWeight: {w_afr_not_recid:.4f}")
print()


# Applying weight factors to classifier
#   Approach 1: applying weights directly
training_set['sample_weights'] = 0.0
training_set.loc[(training_set['race'] == 1) & (training_set['two_year_recid']==1),'sample_weights'] = w_cauc_recid
training_set.loc[(training_set['race'] == 1) & (training_set['two_year_recid']==0),'sample_weights'] = w_cauc_not_recid
training_set.loc[(training_set['race'] == 0) & (training_set['two_year_recid']==1),'sample_weights'] = w_afr_recid
training_set.loc[(training_set['race'] == 0) & (training_set['two_year_recid']==0),'sample_weights'] = w_afr_not_recid

sample_weights = training_set['sample_weights']

classifier_3_1 = LogisticRegression()
classifier_3_1.fit(X_train, Y_train, sample_weight=sample_weights)
test_pred_3_1 = classifier_3_1.predict(X_test)

print("Approach 3.1 - Apply weights directly")
measure(ground_truth=Y_test, prediction=test_pred_3_1, race_group=X_test['race'])

#   Approach 2: re-sampling the dataset
training_set_resampled = training_set.sample(
    frac=1,
    replace=True,
    weights='sample_weights',
    random_state=100
  )

p_res_obs_cauc_recid = ((training_set_resampled['race'] == 1) & (training_set_resampled['two_year_recid'] == 1)).astype(int).mean()
p_res_obs_cauc_not_recid = ((training_set_resampled['race'] == 1) & (training_set_resampled['two_year_recid'] == 0)).astype(int).mean()
p_res_obs_afr_recid = ((training_set_resampled['race'] == 0) & (training_set_resampled['two_year_recid'] == 1)).astype(int).mean()
p_res_obs_afr_not_recid = ((training_set_resampled['race'] == 0) & (training_set_resampled['two_year_recid'] == 0)).astype(int).mean()


X_train_resampled = training_set_resampled.drop(columns=['two_year_recid', 'sample_weights'])
Y_train_resampled = training_set_resampled['two_year_recid']

classifier_3_2 = LogisticRegression()
classifier_3_2.fit(X_train_resampled, Y_train_resampled)
test_pred_3_2 = classifier_3_2.predict(X_test)



print("\n\nApproach 3.2 - Re-sample the dataset")

print("Expected vs Observed Probabilities for resampled dataset\n")
print("Race = 'Caucasian', Recidivate = True")
print(f"\tExpected: {p_exp_cauc_recid:.4f}")
print(f"\tObserved: {p_res_obs_cauc_recid:.4f}")
print()

print("Race = 'Caucasian', Recidivate = False")
print(f"\tExpected: {p_exp_cauc_not_recid:.4f}")
print(f"\tObserved: {p_res_obs_cauc_not_recid:.4f}")
print()

print("Race = 'African-American', Recidivate = True")
print(f"\tExpected: {p_exp_afr_recid:.4f}")
print(f"\tObserved: {p_res_obs_afr_recid:.4f}")
print()


print("Race = 'African-American', Recidivate = False")
print(f"\tExpected: {p_exp_afr_not_recid:.4f}")
print(f"\tObserved: {p_res_obs_afr_not_recid:.4f}")
print()

measure(ground_truth=Y_test, prediction=test_pred_3_2, race_group=X_test['race'])



MARGINAL PROBABILITIES

p(race='Caucasian') = 0.3941
p(race='African-American') = 0.6059
p(two_year_recid=1) = 0.4704
p(two_year_recid=0) = 0.5296


Expected vs Observed Probabilities and Weights

Race = 'Caucasian', Recidivate = True
	Expected: 0.1854
	Observed: 0.1516
	Weight: 1.2230

Race = 'Caucasian', Recidivate = False
	Expected: 0.2087
	Observed: 0.2425
	Weight: 0.8606

Race = 'African-American', Recidivate = True
	Expected: 0.2850
	Observed: 0.3188
	Weight: 0.8940

Race = 'African-American', Recidivate = False
	Expected: 0.3209
	Observed: 0.2871
	Weight: 1.1178

Approach 3.1 - Apply weights directly
accuracy: 0.6448863636363636
precision:0.6508196721311476
recall: 0.7101967799642218
f1: 0.679213002566296

 group: African-American
 statistical parity: 0.6288492706645057
true positive rate: 0.7814569536423841
false positive rate: 0.48253968253968255

 group: Caucasian
 statistical parity: 0.5056947608200456
true positive rate: 0.6264591439688716
false positive rate: 0.33516483516

**Question**

 Report the 4 weights that are used for reweighing and a short **interpretation/discussion** of the weights and the classifier results.
> Answer:
> - w(Race = 'Caucasian', Recidivate = True) = 1.2230
> - w(Race = 'Caucasian', Recidivate = False) = 0.8606
> - w(Race = 'African-American', Recidivate = True) = 0.8940
> - w(Race = 'African-American', Recidivate = False) = 1.1178

**Interpretation**

According to the weights, Caucasians that recidivated and African-Americans that did not recidivate are underrepresented due to having a weight factor > 1.0, whereas combinations with weight factor < 1.0, namely Caucasians that did not recidivate and African Americans that recidivated are overrepresented compared to their expected joint probabilities

During re-weighing we used 2 approaches, directly applying the weights to the classifier (through the .fit method of sklearn) or re-sampling the dataset to match the expected joint probabilities (using pandas' .sample method). Both methods yielded similar results and did overall slightly worse in all four metrics compared to the baseline classifier (1). This is to be exepcted since re-weighing places importance in reducing the overall dependence between the target variable ('two_year_recid') and the protected attribute ('race') instead of optimizing overall accuracy.

If we isolate race groups however, the results are much more interesting. With both approaches, the bias shifted in favor of the 'African-American' group, giving defendants of this group more favorable predictions over the 'Caucasian' groups (which is also indicated by the higher FPR, TPR values of the 'African American' group compared to the 'Caucasian' one). The overall statistical parity gap has now decreased compared to the baseline classifier as well.

Therefore, re-weigihng strategies lead to a less unfair system under the statistical parity gap metric, but does not eradicate unfairness in the system and instead shifts the gap in favor of the other group this time.

#### **5.4 Post-processing: Equalized odds**
Use the predictions by the first classifier for this post processing part (see lecture slides for more information about post processing for equalized odds).

We have the following parameters (A indicates group membership, Y_{hat} the original prediction, Y_{tilde} the prediction of the derived predictor).

* `p_00` = P(Y_{tilde} = 1 | Y_{hat} = 0 & A = 0)
* `p_01` = P(Y_{tilde} = 1 | Y_{hat} = 0 & A = 1)
* `p_10` = P(Y_{tilde} = 1 | Y_{hat} = 1 & A = 0)
* `p_11` = P(Y_{tilde} = 1 | Y_{hat} = 1 & A = 1)


Normally, the best parameters `p_00, p_01, p_10, p_11` are found with a linear program that minimizes loss between predictions of a derived predictor and the actual labels. In this assignment we will not ask you to do this. Instead, we would like you to follow the next steps to find parameters, post-process the data and check the performance of this classifier with post-processing:

1. Generate 5000 different samples of these 4 parameters randomly;
2. Write a function (or more) that applies these 4 parameters to postprocess the predictions.
3. For each generated set of 4 parameters:
  - Change the predicted labels with the function(s) from step 2;
  - Evaluate these 'new' predictions, by calculating group-wise TPR and FPR, as well as overall performance based on F1 and/or accuracy.
4. Choose the best set of parameters. Take into account the equalized odds fairness measure, as well a performance measure like accuracy or F1.
5. Check the overall performance (precision, recall, accuracy, F1, etc.) of the new predictions after post-processing.

In [ ]:
# Your code for step 1
import random

random_parameters = []
for _ in range(1000):
  p_00 = ...
  p_01 = ...
  p_10 = ...
  p_11 = ...
  random_parameters.append({(0, 0): p_00,
                            (0, 1): p_01,
                            (1, 0): p_10,
                            (1, 1): p_11})

# Example, first set of random parameters
print(random_parameters[0])

In [ ]:
# Your code for step 2
# Create a dataframe with the necessary information
df_post_data = pd.DataFrame({'race_num': ...,
                             'pred_labels': ...,
                             'true_labels': ...})

# the number of cases falling in each condition
subset_sizes = {
    (0, 0): len(df_post_data.query('pred_labels == 0 & race_num == 0')),
    (0, 1): len(df_post_data.query('pred_labels == 0 & race_num == 1')),
    (1, 0): len(df_post_data.query('pred_labels == 1 & race_num == 0')),
    (1, 1): len(df_post_data.query('pred_labels == 1 & race_num == 1'))

}

def generate_labels(subset_sizes, p_dict):
    """
    subset_sizes: dict with number of cases falling in each condition
    p_dict: the postprocessing parameters
    """
    new_predictions = {}

    for (prediction, group), p in p_dict.items():

      # The number of instances for which we need to generate labels
      num_instances = subset_sizes[(prediction, group)]

      # Write your code here.

      # save the new predictions
      new_predictions[(prediction, group)] = ...

    return new_predictions

In [ ]:
# Your code for step 3

for p_dict in random_parameters:

  new_predictions = generate_labels(subset_sizes, p_dict)

  # replace the predictions
  df_copy = df_post_data.copy()

  for (pred, group), p in p_dict.items():

    new_preds = new_predictions[(pred,group)]
    df_copy.loc[(df_post_data['pred_labels'] == pred) &
                (df_post_data['race_num'] == group), 'pred_labels'] = new_preds


  # evaluate the new predictions and save the scores
  # Write your code here.


In [ ]:
# Your code for step 4 and 5


**Question**

Describe how you selected the best set of parameters. Furthermore, how do you interpret the best set of parameters that you found? And what do you think of the results of the new classifier?
>Answer

#### **Overall discussion**
For all 4 classifiers that you trained, describe:
- Does this classifier satisfies statistical parity?
- Does the classifier satisfy the equal opportunity criterion?

Finally, how do the different classifiers compare against each other?

>Answer

### **6. Intersectional fairness**
In the questions above `race` was the only protected attribute. However, multiple protected attributes sometimes interact, leading to different fairness outcomes for different combinations of these protected attributes.

Now explore the intersectional fairness for protected attributes `race` and `sex` for the first two classifiers from question 5. Make a combination of the `race` and `sex` column, resulting in four new subgroups (e.g., female Caucasian), and report the maximum difference between the subgroups for statistical parity, TPR and FPR.
For example, suppose we have four groups with TPRs 0.1, 0.2, 0.3, 0.8, then the maximum difference is 0.7.

Your code to evaluate intersectional fairness for Classifier 1:




In [62]:
# Your code for intersectional fairness
c1_intersec_results = pd.DataFrame({
    "race" : X_test['race'],
    "sex" : X_test['sex'],
    "prediction" : test_pred_1,
    "ground_truth" : Y_test
}).reset_index(drop=True)

print("Classifier 1\n")

c1_intersec_results['race'] = c1_intersec_results['race'].replace(0, 'African-American')
c1_intersec_results['race'] = c1_intersec_results['race'].replace(1, 'Caucasian')

c1_intersec_results['sex'] = c1_intersec_results['sex'].replace(0, 'Female')
c1_intersec_results['sex'] = c1_intersec_results['sex'].replace(1, 'Male')

c1_intersec_results['FP'] = (c1_intersec_results['prediction'] == 0) & (c1_intersec_results['ground_truth'] == 1).astype(int)
c1_intersec_results['FN'] = (c1_intersec_results['prediction'] == 1) & (c1_intersec_results['ground_truth'] == 0).astype(int)
c1_intersec_results['TP'] = (c1_intersec_results['prediction'] == 0) & (c1_intersec_results['ground_truth'] == 0).astype(int)
c1_intersec_results['TN'] = (c1_intersec_results['prediction'] == 1) & (c1_intersec_results['ground_truth'] == 1).astype(int)

intersec_metrics_1 = c1_intersec_results.groupby(["race","sex"]).agg(
  FP = ("FP", "sum"),
  FN = ("FN", "sum"),
  TP = ("TP", "sum"),
  TN = ("TN", "sum"),
  Base_rate = ("prediction", lambda x: 1-x.mean())
).reset_index()

intersec_metrics_1['TP_RATE'] = intersec_metrics_1.TP / (intersec_metrics_1.TP + intersec_metrics_1.FN)
intersec_metrics_1['FP_RATE'] = intersec_metrics_1.FP / (intersec_metrics_1.FP + intersec_metrics_1.TN)
display(intersec_metrics_1[['race','sex', 'TP_RATE', 'FP_RATE', 'Base_rate']])
print(f"\n\nBase rate Maximum Difference: {intersec_metrics_1['Base_rate'].max() - intersec_metrics_1['Base_rate'].min():.2f}")
print(f"TPR Maximum Difference: {intersec_metrics_1['TP_RATE'].max() - intersec_metrics_1['TP_RATE'].min():.2f}")
print(f"FPR Maximum Difference: {intersec_metrics_1['FP_RATE'].max() - intersec_metrics_1['FP_RATE'].min():.2f}")


Classifier 1



,race,sex,TP_RATE,FP_RATE,Base_rate
0,African-American,Female,0.934426,0.627907,0.807692
1,African-American,Male,0.568465,0.279412,0.415205
2,Caucasian,Female,0.945455,0.800000,0.894118
3,Caucasian,Male,0.826733,0.546053,0.706215




Base rate Maximum Difference: 0.48
TPR Maximum Difference: 0.38
FPR Maximum Difference: 0.52


Your code to evaluate intersectional fairness for Classifier 2:


In [63]:
# Your code for intersectional fairness
c2_intersec_results = pd.DataFrame({
    "race" : X_test['race'],
    "sex" : X_test['sex'],
    "prediction" : test_pred_2,
    "ground_truth" : Y_test
}).reset_index(drop=True)

print("Classifier 2\n")

c2_intersec_results['race'] = c2_intersec_results['race'].replace(0, 'African-American')
c2_intersec_results['race'] = c2_intersec_results['race'].replace(1, 'Caucasian')

c2_intersec_results['sex'] = c2_intersec_results['sex'].replace(0, 'Female')
c2_intersec_results['sex'] = c2_intersec_results['sex'].replace(1, 'Male')

c2_intersec_results['FP'] = (c2_intersec_results['prediction'] == 0) & (c2_intersec_results['ground_truth'] == 1).astype(int)
c2_intersec_results['FN'] = (c2_intersec_results['prediction'] == 1) & (c2_intersec_results['ground_truth'] == 0).astype(int)
c2_intersec_results['TP'] = (c2_intersec_results['prediction'] == 0) & (c2_intersec_results['ground_truth'] == 0).astype(int)
c2_intersec_results['TN'] = (c2_intersec_results['prediction'] == 1) & (c2_intersec_results['ground_truth'] == 1).astype(int)

intersec_metrics_2 = c2_intersec_results.groupby(["race","sex"]).agg(
  FP = ("FP", "sum"),
  FN = ("FN", "sum"),
  TP = ("TP", "sum"),
  TN = ("TN", "sum"),
  Base_rate = ("prediction", lambda x: 1-x.mean())
).reset_index()

intersec_metrics_2['TP_RATE'] = intersec_metrics_2.TP / (intersec_metrics_2.TP + intersec_metrics_2.FN)
intersec_metrics_2['FP_RATE'] = intersec_metrics_2.FP / (intersec_metrics_2.FP + intersec_metrics_2.TN)
display(intersec_metrics_2[['race','sex', 'TP_RATE', 'FP_RATE', 'Base_rate']])
print(f"\n\nBase rate Maximum Difference: {intersec_metrics_2['Base_rate'].max() - intersec_metrics_2['Base_rate'].min():.2f}")
print(f"TPR Maximum Difference: {intersec_metrics_2['TP_RATE'].max() - intersec_metrics_2['TP_RATE'].min():.2f}")
print(f"FPR Maximum Difference: {intersec_metrics_2['FP_RATE'].max() - intersec_metrics_2['FP_RATE'].min():.2f}")


Classifier 2



,race,sex,TP_RATE,FP_RATE,Base_rate
0,African-American,Female,0.934426,0.627907,0.807692
1,African-American,Male,0.585062,0.290441,0.428850
2,Caucasian,Female,0.945455,0.766667,0.882353
3,Caucasian,Male,0.777228,0.500000,0.658192




Base rate Maximum Difference: 0.45
TPR Maximum Difference: 0.36
FPR Maximum Difference: 0.48


**Question**

Write down a short interpretation of the results you calculated. What do you see?
> Answer:

> * African American Males receive the fewest favorable predictions and Caucasian Females receive the most. The largest gaps across all 3 metrics (statistical parity / FPR / TPR) are betweeen subgroups 'Caucasian / Female'  and 'African-American / Male'.
> * Intersectional gaps are much larger vs when only considering gaps between race groups (e.g here: statistical parity gap 0.48 and 0.45 respectively vs 0.26 and 0.21 when considering the 'race' attribute only).
> * Classifier 2 slightly reduces disparity, but large gaps across all metrics still perist
> * Gender disparities appear much stronger than race disparities and female defendants are much more likely to receive favorable predictions compared to male defendants.




## Discussion
Provide a short ethical discussion (1 or 2 paragraphs) reflecting on these two aspects:

1) The use of a ML system to try to predict recidivism;

2) The public release of a dataset like this.

> Answer

> A ML-based system that predicts recidivism could be a useful tool since it can give an estimate of a defendants likelihood of re-offending and help shape final sentence accordingly. However, if said system is trained on data that reflects structural inequalities in society it can reproduce or even amplify these inequalities in its predictions. This can create a feedback loop: biases reflected in data can create a system which prodices biased recidivism predictions, leading to more unfar sentencing etc, which leads to an even more unfair society. Therefore such system should be treated with great caution, both in how it is trained and in how its results are interpreted. The outcome of such system should only be utilized as decision support tool and not viewed as objective truth.

> The release of such dataset is very important in allowing independent researchers to inspect the data and produce their own analyses and models and see whether biases in data lead to biases in model outputs. This transparency is very important for systems used in high-stakes areas like the criminal justice system. It allows for critique and holds the creators of such systems accoutnable. However releasing such data should also be done with caution because of the sensitive personal  information that they include.